# 02 - Benchmarks and TimeMixer on the single chronological split

Fits the benchmark models and TimeMixer on the training block, tunes TimeMixer on the
validation block, and reports the held-out test block once.

Models: persistence, HAR+IV (linear and log), GARCH(1,1), TimeMixer ensemble.
Metrics: RMSE, MAE, bias, QLIKE; Diebold-Mariano tests on the validation block.

**Needs:** outputs of `01_data_preparation.ipynb`.
**Writes:** `results/val_predictions.csv`, `results/test_predictions.csv`, `results/results_table.csv`.

In [ ]:
import copy
import os
import sys
from pathlib import Path
from types import SimpleNamespace

import arch
import numpy as np
import pandas as pd
import statsmodels.api as sm
import torch
import torch.nn as nn
from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error

# The notebooks live in notebooks/, but all paths below are relative to the repository root.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

# TimeMixer reference implementation (git submodule in TimeMixer/)
sys.path.insert(0, "TimeMixer")
from models.TimeMixer import Model  # noqa: E402

## 0. Load data and rebuild the split

In [ ]:
model_df = pd.read_csv("data/eurusd_model_ready.csv", parse_dates=["Date"])

# Same 70/15/15 split with 22-day gaps as in 01_data_preparation
n = len(model_df)
train_end, val_end, gap = int(n * 0.70), int(n * 0.85), 22
train = model_df.iloc[:train_end]
val = model_df.iloc[train_end + gap : val_end]
test = model_df.iloc[val_end + gap :]

# Standardised 64-day windows and target scaling
d = np.load("data/windows_N64.npz", allow_pickle=True)
Xtr, ytr, Xva, yva, Xte, yte = [d[k] for k in ["Xtr", "ytr", "Xva", "yva", "Xte", "yte"]]
y_mu, y_sd = float(d["y_mu"]), float(d["y_sd"])

# Daily returns on the full sample, used to fit GARCH
returns = pd.read_csv("data/eurusd_returns_full.csv", parse_dates=["Date"]).set_index("Date")[
    "log_return"
]

## 1. Benchmarks on the validation block

Target: realized volatility over the next 22 trading days (annualised %).

In [1]:
def score(y, pred):
    """(RMSE, MAE) rounded to 3 decimals."""
    return (
        round(float(np.sqrt(mean_squared_error(y, pred))), 3),
        round(float(mean_absolute_error(y, pred)), 3),
    )


y_tr, y_va = train["target_22d"], val["target_22d"]

# 1. Persistence: next month's volatility equals this month's
print("persistence (RMSE, MAE):", score(y_va, val["realized_vol_22d"]))

# 2. HAR: daily, weekly and monthly range-based components
har = ["park_d", "park_w", "park_m"]
m1 = sm.OLS(y_tr, sm.add_constant(train[har])).fit()
print("HAR:", score(y_va, m1.predict(sm.add_constant(val[har]))))

# 3. HAR + one-month implied volatility
har_iv = har + ["IMPL_VOL_1M"]
m2 = sm.OLS(y_tr, sm.add_constant(train[har_iv])).fit()
print("HAR+IV:", score(y_va, m2.predict(sm.add_constant(val[har_iv]))))
print(m2.params.round(3))

persistence (RMSE, MAE): (2.538, 2.152)
HAR: (1.865, 1.687)
HAR+IV: (1.77, 1.549)
const          0.749
park_d        -0.000
park_w        -0.072
park_m         0.121
IMPL_VOL_1M    0.852
dtype: float64


In [2]:
# 4. Log-log HAR + implied volatility (forecasts are strictly positive)
cols = ["park_d", "park_w", "park_m", "IMPL_VOL_1M"]
print("min values in train (must be > 0):", train[cols].min().round(3).to_dict())

Xtr = np.log(train[cols])
Xva = np.log(val[cols])
m3 = sm.OLS(np.log(train["target_22d"]), sm.add_constant(Xtr)).fit()
pred = np.exp(m3.predict(sm.add_constant(Xva)))

print("HAR+IV log:", score(y_va, pred))
print("mean error:", round((pred - y_va).mean(), 3))
print(m3.params.round(3))

min values in train (must be > 0): {'park_d': 0.552, 'park_w': 2.802, 'park_m': 3.767, 'IMPL_VOL_1M': 4.738}
HAR+IV log: (np.float64(1.679), 1.461)
mean error: 0.879
const          0.183
park_d         0.003
park_w        -0.066
park_m         0.193
IMPL_VOL_1M    0.772
dtype: float64


In [3]:
# 5. GARCH(1,1) fitted on data up to the end of the training block
import arch

r = returns * 100  # percent returns
am = arch.arch_model(r, mean="Constant", vol="GARCH", p=1, q=1, dist="normal")
res = am.fit(last_obs=train["Date"].iloc[-1], disp="off")
print(res.params.round(4))

fc = res.forecast(horizon=22, start=val["Date"].iloc[0], reindex=False)
garch_pred = np.sqrt(fc.variance.mean(axis=1) * 252)  # annualized %, same units as target
garch_pred = garch_pred.reindex(val["Date"])

print("NaNs:", garch_pred.isnull().sum())
print("GARCH:", score(val["target_22d"].values, garch_pred.values))

mu         -0.0078
omega       0.0023
alpha[1]    0.0325
beta[1]     0.9586
Name: params, dtype: float64
NaNs: 0
GARCH: (np.float64(2.344), 2.211)


In [4]:
# Mean forecast error (bias) of the two benchmarks
print("mean error (pred - actual):")
print("GARCH:", round((garch_pred.values - val["target_22d"].values).mean(), 3))
print("HAR+IV:", round((m2.predict(sm.add_constant(val[har_iv])) - y_va).mean(), 3))
print("mean target in val:", round(y_va.mean(), 3))

mean error (pred - actual):
GARCH: 1.685
HAR+IV: 1.031
mean target in val: 6.305


### Regime check

The average target and implied volatility differ a lot between blocks (volatility falls over
the sample), which matters for the interpretation of bias later on.

In [5]:
for name, d in [("train", train), ("val", val), ("test", test)]:
    print(
        name,
        "target:",
        round(d["target_22d"].mean(), 2),
        "| impl vol:",
        round(d["IMPL_VOL_1M"].mean(), 2),
    )

train target: 7.62 | impl vol: 7.66
val target: 6.31 | impl vol: 7.3
test target: 5.54 | impl vol: 5.93


## 2. TimeMixer

A small TimeMixer core (7 inputs, 64-day window) with a linear read-out `Linear(7, 1)`,
trained on log-standardised targets with MSE loss, Adam, early stopping on the validation loss.

In [6]:
# Base configuration (d_model 16); smaller variants are compared below
cfg = SimpleNamespace(
    task_name="long_term_forecast",
    seq_len=64,
    label_len=0,
    pred_len=1,
    enc_in=7,
    c_out=7,
    d_model=16,
    d_ff=32,
    e_layers=2,
    dropout=0.2,
    down_sampling_layers=3,
    down_sampling_window=2,
    down_sampling_method="avg",
    channel_independence=0,
    use_norm=0,
    decomp_method="moving_avg",
    moving_avg=5,
    top_k=5,
    embed="timeF",
    freq="d",
    use_future_temporal_feature=0,
    num_class=0,
)

torch.manual_seed(0)
core = Model(cfg)
xb = torch.tensor(Xtr[:8], dtype=torch.float32)
out = core(xb, None, None, None)
print("output shape:", tuple(out.shape))
print("parameters:", sum(p.numel() for p in core.parameters()))

output shape: (8, 1, 7)
parameters: 35583


In [7]:
class Net(nn.Module):
    def __init__(self, core):
        super().__init__()
        self.core = core
        self.readout = nn.Linear(7, 1)

    def forward(self, x):
        z = self.core(x, None, None, None)  # (B, 1, 7)
        return self.readout(z[:, 0, :]).squeeze(-1)  # (B,)


def to_t(a):
    return torch.tensor(a, dtype=torch.float32)


Xtr_t, ytr_t = to_t(Xtr), to_t(ytr)
Xva_t, yva_t = to_t(Xva), to_t(yva)


def train_one(seed, c=cfg, window=64, max_epochs=100, patience=10, lr=1e-3, wd=1e-4, bs=32):
    assert c.seq_len == window, "cfg.seq_len must match the window"
    Xt, Xv = Xtr_t[:, -window:, :], Xva_t[:, -window:, :]
    torch.manual_seed(seed)
    np.random.seed(seed)
    net = Net(Model(c))
    opt = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=wd)
    loss_fn = nn.MSELoss()
    best, best_state, bad = float("inf"), None, 0
    for epoch in range(max_epochs):
        net.train()
        perm = torch.randperm(len(Xt))
        for i in range(0, len(perm), bs):
            idx = perm[i : i + bs]
            opt.zero_grad()
            loss_fn(net(Xt[idx]), ytr_t[idx]).backward()
            opt.step()
        net.eval()
        with torch.no_grad():
            v = loss_fn(net(Xv), yva_t).item()
        if v < best - 1e-5:
            best, bad = v, 0
            best_state = {k: x.clone() for k, x in net.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    if best_state is not None:
        net.load_state_dict(best_state)
    return net, best, epoch + 1


def predict(net, X, window=64):
    # back to vol units (undo scaling + log)
    net.eval()
    with torch.no_grad():
        return np.exp(net(X[:, -window:, :]).numpy() * y_sd + y_mu)


net, best_val, n_ep = train_one(seed=0)
pred = predict(net, Xva_t)

print("epochs run:", n_ep, "| best val loss (scaled):", round(best_val, 3))
print("TimeMixer val (RMSE, MAE):", score(y_va, pred))
print("mean error:", round((pred - y_va.values).mean(), 3))

epochs run: 13 | best val loss (scaled): 0.919
TimeMixer val (RMSE, MAE): (np.float64(1.806), 1.528)
mean error: 0.784


### Seed variability and configuration search (validation block only)

In [8]:
# Five seeds with the base configuration
rows, preds = [], []
for s in range(5):
    net_s, bv, ne = train_one(seed=s)
    ps = predict(net_s, Xva_t)
    preds.append(ps)
    rmse, mae = score(y_va, ps)
    rows.append((s, ne, rmse, mae, round((ps - y_va.values).mean(), 3)))
    print("seed", s, "| epochs", ne, "| RMSE", rmse, "| MAE", mae, "| bias", rows[-1][4])

seed_rmse = np.array([x[2] for x in rows], dtype=float)
print("RMSE mean:", round(seed_rmse.mean(), 3), "| std:", round(seed_rmse.std(), 3))
print("ensemble of 5 (RMSE, MAE):", score(y_va, np.mean(preds, axis=0)))

seed 0 | epochs 13 | RMSE 1.806 | MAE 1.528 | bias 0.784
seed 1 | epochs 14 | RMSE 1.943 | MAE 1.758 | bias 1.034
seed 2 | epochs 27 | RMSE 2.035 | MAE 1.684 | bias 0.748
seed 3 | epochs 23 | RMSE 1.71 | MAE 1.501 | bias 0.951
seed 4 | epochs 19 | RMSE 1.737 | MAE 1.505 | bias 0.684
RMSE mean: 1.846 | std: 0.124
ensemble of 5 (RMSE, MAE): (np.float64(1.757), 1.558)


In [9]:
def run_cfg(name, window=64, seeds=5, **over):
    c = copy.deepcopy(cfg)
    for k, v in over.items():
        setattr(c, k, v)
    c.seq_len = window
    rms, ps_all, biases = [], [], []
    for s in range(seeds):
        net_s, _, _ = train_one(s, c=c, window=window)
        ps = predict(net_s, Xva_t, window)
        ps_all.append(ps)
        rms.append(score(y_va, ps)[0])
        biases.append((ps - y_va.values).mean())
    ens = score(y_va, np.mean(ps_all, axis=0))
    print(
        f"{name:10s} RMSE mean {np.mean(rms):.3f} std {np.std(rms):.3f} | "
        f"ensemble RMSE {ens[0]:.3f} MAE {ens[1]:.3f} | mean bias {np.mean(biases):.3f}"
    )
    return ps_all


# each config trains 5 seeds -> takes a few minutes
cfg_preds = {}
cfg_preds["base"] = run_cfg("base")
cfg_preds["A small"] = run_cfg("A small", d_model=8, d_ff=16)
cfg_preds["B dropout"] = run_cfg("B dropout", dropout=0.4)
cfg_preds["C window32"] = run_cfg("C window32", window=32)

base       RMSE mean 1.846 std 0.124 | ensemble RMSE 1.757 MAE 1.558 | mean bias 0.840
A small    RMSE mean 1.747 std 0.070 | ensemble RMSE 1.577 MAE 1.340 | mean bias 0.703
B dropout  RMSE mean 1.837 std 0.171 | ensemble RMSE 1.737 MAE 1.489 | mean bias 0.719
C window32 RMSE mean 1.953 std 0.119 | ensemble RMSE 1.913 MAE 1.693 | mean bias 1.049


### Chosen configuration and validation tests

Config A (`d_model=8`, `d_ff=16`) is kept, with a 10-seed ensemble. The Diebold-Mariano
statistic below uses a 21-lag Bartlett HAC variance (no small-sample correction here; the
corrected tests are in notebook 03).

In [10]:
preds_A = run_cfg("A small x10", seeds=10, d_model=8, d_ff=16)
p_tm = np.mean(preds_A, axis=0)

y = y_va.values
cands = {
    "GARCH": garch_pred.values,
    "HAR+IV (lin)": m2.predict(sm.add_constant(val[har_iv])).values,
    "HAR+IV (log)": np.exp(m3.predict(sm.add_constant(np.log(val[cols])))).values,
    "TimeMixer ens10": p_tm,
}


def qlike(y, p):
    r = (y**2) / (p**2)
    return np.mean(r - np.log(r) - 1)


def dm_hac(y, p1, p2, lags=21):
    d = (y - p1) ** 2 - (y - p2) ** 2  # positive stat => model 2 is better
    T = len(d)
    e = d - d.mean()
    var = (e @ e) / T
    for k in range(1, lags + 1):
        var += 2 * (1 - k / (lags + 1)) * (e[k:] @ e[:-k]) / T
    stat = d.mean() / np.sqrt(var / T)
    return stat, 2 * (1 - stats.norm.cdf(abs(stat)))


print(f"{'model':16s} {'RMSE':>6s} {'MAE':>6s} {'bias':>6s} {'QLIKE':>7s}")
for k, p in cands.items():
    rm, ma = score(y, p)
    print(f"{k:16s} {rm:6.3f} {ma:6.3f} {(p - y).mean():6.3f} {qlike(y, p):7.4f}")

for k in ["GARCH", "HAR+IV (lin)", "HAR+IV (log)"]:
    s, pv = dm_hac(y, cands[k], p_tm)
    print(f"DM {k} vs TimeMixer: stat {s:.2f}, p {pv:.3f}")

A small x10 RMSE mean 1.740 std 0.153 | ensemble RMSE 1.589 MAE 1.417 | mean bias 0.801
model              RMSE    MAE   bias   QLIKE
GARCH             2.344  2.211  1.685  0.2199
HAR+IV (lin)      1.770  1.549  1.031  0.1521
HAR+IV (log)      1.679  1.461  0.879  0.1400
TimeMixer ens10   1.589  1.417  0.801  0.1228
DM GARCH vs TimeMixer: stat 5.24, p 0.000
DM HAR+IV (lin) vs TimeMixer: stat 1.51, p 0.130
DM HAR+IV (log) vs TimeMixer: stat 0.91, p 0.360


## 3. Held-out test block

Evaluated once, with the configuration fixed on the validation block.

In [11]:
def run_final(seeds=10, **over):
    c = copy.deepcopy(cfg)
    for k, v in over.items():
        setattr(c, k, v)
    Xte_t = to_t(Xte)
    pv, pt = [], []
    for s in range(seeds):
        torch.manual_seed(s)
        np.random.seed(s)
        net = Net(Model(c))
        opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-4)
        loss_fn = nn.MSELoss()
        best, best_state, bad = float("inf"), None, 0
        for epoch in range(100):
            net.train()
            perm = torch.randperm(len(Xtr_t))
            for i in range(0, len(perm), 32):
                idx = perm[i : i + 32]
                opt.zero_grad()
                loss_fn(net(Xtr_t[idx]), ytr_t[idx]).backward()
                opt.step()
            net.eval()
            with torch.no_grad():
                v = loss_fn(net(Xva_t), yva_t).item()
            if v < best - 1e-5:
                best, bad = v, 0
                best_state = {k: x.clone() for k, x in net.state_dict().items()}
            else:
                bad += 1
                if bad >= 10:
                    break
        net.load_state_dict(best_state)
        net.eval()
        with torch.no_grad():
            pv.append(np.exp(net(Xva_t).numpy() * y_sd + y_mu))
            pt.append(np.exp(net(Xte_t).numpy() * y_sd + y_mu))
    return pv, pt


pv, pt = run_final(seeds=10, d_model=8, d_ff=16)
print("val ensemble RMSE (should match 1.589):", score(y_va, np.mean(pv, axis=0))[0])

y_te = test["target_22d"].values
fc_te = res.forecast(horizon=22, start=test["Date"].iloc[0], reindex=False)
g_te = np.sqrt(fc_te.variance.mean(axis=1) * 252).reindex(test["Date"]).values
print(
    "GARCH NaNs on test:",
    int(np.isnan(g_te).sum()),
    "| mean target on test:",
    round(y_te.mean(), 2),
)

cands_te = {
    "Persistence": test["realized_vol_22d"].values,
    "GARCH": g_te,
    "HAR+IV (lin)": m2.predict(sm.add_constant(test[har_iv])).values,
    "HAR+IV (log)": np.exp(m3.predict(sm.add_constant(np.log(test[cols])))).values,
    "TimeMixer ens10": np.mean(pt, axis=0),
}

print(f"{'model':16s} {'RMSE':>6s} {'MAE':>6s} {'bias':>6s} {'QLIKE':>7s}")
for k, p in cands_te.items():
    rm, ma = score(y_te, p)
    print(f"{k:16s} {rm:6.3f} {ma:6.3f} {(p - y_te).mean():6.3f} {qlike(y_te, p):7.4f}")

single = [score(y_te, p)[0] for p in pt]
print(
    "TimeMixer single-seed test RMSE: mean",
    round(np.mean(single), 3),
    "std",
    round(np.std(single), 3),
)

for k in ["GARCH", "HAR+IV (lin)", "HAR+IV (log)"]:
    s, pv_ = dm_hac(y_te, cands_te[k], cands_te["TimeMixer ens10"])
    print(f"DM {k} vs TimeMixer: stat {s:.2f}, p {pv_:.3f}")

val ensemble RMSE (should match 1.589): 1.589
GARCH NaNs on test: 0 | mean target on test: 5.54
model              RMSE    MAE   bias   QLIKE
Persistence       2.169  1.580  0.090  0.4253
GARCH             1.919  1.751  0.946  0.1917
HAR+IV (lin)      1.645  1.408  0.517  0.1479
HAR+IV (log)      1.634  1.351  0.363  0.1534
TimeMixer ens10   1.819  1.471  0.117  0.2070
TimeMixer single-seed test RMSE: mean 1.913 std 0.282
DM GARCH vs TimeMixer: stat 0.49, p 0.627
DM HAR+IV (lin) vs TimeMixer: stat -1.01, p 0.311
DM HAR+IV (log) vs TimeMixer: stat -1.24, p 0.217


In [12]:
# Save validation and test forecasts for the later notebooks and figures
os.makedirs("results", exist_ok=True)

pd.DataFrame({"Date": test["Date"].values, "actual": y_te, **cands_te}).to_csv(
    "results/test_predictions.csv", index=False
)
pd.DataFrame({"Date": val["Date"].values, "actual": y, **cands}).to_csv(
    "results/val_predictions.csv", index=False
)

print(os.listdir("results"))

['val_predictions.csv', 'test_predictions.csv']


## 4. Results table and forecast calibration

In [13]:
val = pd.read_csv("results/val_predictions.csv", parse_dates=["Date"])
test = pd.read_csv("results/test_predictions.csv", parse_dates=["Date"])


def table(df):
    y = df["actual"].values
    rows = []
    for c in [c for c in df.columns if c not in ("Date", "actual")]:
        p = df[c].values
        rows.append(
            {
                "model": c,
                "RMSE": np.sqrt(np.mean((y - p) ** 2)),
                "MAE": np.mean(np.abs(y - p)),
                "bias": np.mean(p - y),
                "QLIKE": qlike(y, p),
            }
        )
    return pd.DataFrame(rows).set_index("model").round(3)


tv, tt = table(val), table(test)
print("VALIDATION\n", tv, "\n\nTEST\n", tt)

pd.concat({"validation": tv, "test": tt}, axis=1).to_csv("results/results_table.csv")

VALIDATION
                   RMSE    MAE   bias  QLIKE
model                                      
GARCH            2.344  2.211  1.685  0.220
HAR+IV (lin)     1.770  1.549  1.031  0.152
HAR+IV (log)     1.679  1.461  0.879  0.140
TimeMixer ens10  1.589  1.417  0.801  0.123 

TEST
                   RMSE    MAE   bias  QLIKE
model                                      
Persistence      2.169  1.580  0.090  0.425
GARCH            1.919  1.751  0.946  0.192
HAR+IV (lin)     1.645  1.408  0.517  0.148
HAR+IV (log)     1.634  1.351  0.363  0.153
TimeMixer ens10  1.819  1.471  0.117  0.207


In [14]:
# Correlation and calibration slope (actual regressed on forecast)
for name, df in [("VALIDATION", val), ("TEST", test)]:
    print(name)
    y = df["actual"].values
    for c in [c for c in df.columns if c not in ("Date", "actual")]:
        p = df[c].values
        slope = np.polyfit(p, y, 1)[0]
        print(f"  {c:16s} corr {np.corrcoef(p, y)[0, 1]:6.3f} | slope {slope:6.3f}")

VALIDATION
  GARCH            corr  0.614 | slope  0.935
  HAR+IV (lin)     corr  0.750 | slope  1.425
  HAR+IV (log)     corr  0.750 | slope  1.387
  TimeMixer ens10  corr  0.766 | slope  1.299
TEST
  Persistence      corr  0.084 | slope  0.077
  GARCH            corr  0.022 | slope  0.047
  HAR+IV (lin)     corr  0.215 | slope  0.417
  HAR+IV (log)     corr  0.175 | slope  0.338
  TimeMixer ens10  corr -0.184 | slope -0.381
